In [4]:
import torch
import math
import itertools
import numpy as np
from sklearn.datasets import make_regression

# 1. Przygotowanie danych (symulujemy "toy set" dla N=5 cech, aby obliczenia były szybkie)
N_FEATURES = 5
X_np, y_np = make_regression(n_samples=1000, n_features=N_FEATURES)
X = torch.tensor(X_np, dtype=torch.float32)

# Symulujemy prosty wytrenowany model regresyjny dla potrzeb zadania
class DummyModel:
    def __call__(self, x):
        # Zakładamy, że model mnoży cechy przez stałe wagi
        weights = torch.tensor([1.5, -2.0, 3.0, 0.0, 0.5])
        return torch.sum(x * weights, dim=1)

model = DummyModel()

# 2. Definiujemy dane wejściowe do wyjaśnienia (zgodnie z wytycznymi)
X_to_explain = X[0]               # Wyjaśniamy pierwszą próbkę ze zbioru
means = X.mean(dim=0)             # Wartości domyślne to średnie wartości cech
N = N_FEATURES
N_set = set(range(N))             # Zbiór wszystkich indeksów cech {0, 1, ..., N-1}

shap_values = np.zeros(N)

# 3. Implementacja definicyjna wartości Shapleya
for i in range(N):
    # Zbiór N/i (wszystkie cechy z usuniętą zmienną i)
    N_minus_i = list(N_set - {i}) 
    marginal_contribution_sum = 0
    
    # Przechodzimy przez wszystkie możliwe rozmiary podzbiorów S (od 0 do N-1)
    for subset_size in range(N):
        
        # Generujemy wszystkie możliwe podzbiory S o danym rozmiarze ze zbioru N/i
        for S in itertools.combinations(N_minus_i, subset_size):
            S = list(S)
            
            # Obliczamy wagę kombinatoryczną ze wzoru: |S|! * (|N| - |S| - 1)! / |N|!
            weight = (math.factorial(len(S)) * math.factorial(N - len(S) - 1)) / math.factorial(N)
            
            # Tworzymy wejście v(S) -> cechy z S mają oryginalną wartość, reszta to wartości domyślne
            mask_S = torch.zeros(N)
            mask_S[S] = 1
            input_S = X_to_explain * mask_S + means * (1 - mask_S)
            
            # Tworzymy wejście v(S U {i}) -> to samo co wyżej, ale dodatkowo włączamy cechę 'i'
            mask_S_i = mask_S.clone()
            mask_S_i[i] = 1
            input_S_i = X_to_explain * mask_S_i + means * (1 - mask_S_i)
            
            # Obliczamy odpowiedź modelu dla obu wejść (dodajemy wymiar batcha przez unsqueeze)
            v_S_i = model(input_S_i.unsqueeze(0)).item()
            v_S = model(input_S.unsqueeze(0)).item()
            
            # Dodajemy ważoną różnicę (wkład marginalny) do ogólnej sumy dla cechy i
            marginal_contribution_sum += weight * (v_S_i - v_S)
            
    # Zapisujemy wyznaczoną dokładną wartość Shapleya dla cechy i
    shap_values[i] = marginal_contribution_sum

# 4. Wyświetlenie wyników
print("Dokładne wartości Shapleya dla poszczególnych cech próbki X_to_explain:")
for idx, val in enumerate(shap_values):
    print(f"Cecha {idx}: {val:.4f}")

Dokładne wartości Shapleya dla poszczególnych cech próbki X_to_explain:
Cecha 0: 2.5042
Cecha 1: -0.1934
Cecha 2: 6.8898
Cecha 3: 0.0000
Cecha 4: -0.6852
